# Stage C & D: Multimodal Waveform Debugging & Layout Synthesis 📐

**TL;DR:** Drawing layout is pure geometry. But the AI cannot "see" like a human does. It only sees text. We have to teach an Agent how to write TCL commands for tools like **ALIGN** or **Magic VLSI** to draw rectangles, run Design Rule Checks (DRC), and iteratively adjust placement (Stage D). Additionally, consider integrating Multimodal Vision tools (Stage C) to interpret graphical waveform ringing instead of just text logs.

---

## 🛠️ The Challenge of Layout for LLMs

If you tell an LLM: "Draw an NMOS transistor."

It cannot draw it. It must write a script that tells a layout tool to draw it.
```tcl
# Magic VLSI Example Command
box 0 0 10 2    # Draw the diffusion layer
paint ndiff
box 4 -2 6 4    # Draw the poly gate
paint poly
```

### Design Rule Checks (DRC)
But what if the poly gate is too close to the diffusion edge? Magic will output a **DRC Error**: `Poly spacing to N-diffusion must be > 2 lambda`.

We need a Multi-Agent system where:
1. **Coder** writes the TCL script.
2. **Simulator** runs `magic -dnull -noconsole < script.tcl`.
3. **Critic** reads the `drc list` output and rewrites the script to add spacing.

In [ ]:
# !pip install langgraph langchain langchain-openai

import os
import subprocess
import re
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage, SystemMessage

# Set your API Key here
# os.environ["OPENAI_API_KEY"] = "sk-..."

## 1. Stage C: Multimodal Waveform Debugger (Theory Concept)
Before layout, what if the design has ringing or instability that isn't easily caught by a single printed string log? We can pass a generated `.png` plot to a Vision LLM to act as our `Critic`.

In [ ]:
from langchain_core.messages import HumanMessage

def visual_critic_agent(image_path: str):
    """An agent that 'looks' at a SPICE transient plot and diagnoses issues."""
    # Note: Requires a vision-capable model like gpt-4o or claude-3-5-sonnet
    llm = ChatOpenAI(model="gpt-4o", max_tokens=256)
    
    import base64
    with open(image_path, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode('utf-8')
        
    message = HumanMessage(
        content=[
            {"type": "text", "text": "You are an Analog IC engineer. Analyze this transient response plot. Do you see instability, ringing, or clipping? If so, recommend a fix (e.g., increase compensation capacitor)."},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_image}"}}
        ]
    )
    
    # return llm.invoke([message]).content
    return "(Mocked) I detect severe ringing. Please increase the Miller compensation capacitor by 1pF."


## 2. Define the Shared State (Stage D Layout Version)

In [ ]:
class LayoutDesignState(TypedDict):
    target_component: str
    current_tcl_script: str
    drc_errors: List[str]
    iterations: int
    status: str

## 3. The Tools (Magic VLSI DRC Checker for Stage D)
This simulates running an open-source layout tool (like Magic or ALIGN) in batch mode. 
*(Note: Since this notebook assumes you might not have Magic installed, we mock the tool output to demonstrate the agentic logic.)*

In [ ]:
def mock_magic_drc(tcl_script: str) -> List[str]:
    """Simulates running Magic VLSI and parsing DRC errors."""
    errors = []
    # If the script doesn't have a specific spacing command, we force a mock DRC error
    if "move e 2" not in tcl_script and "box 4 -2" in tcl_script:
        errors.append("DRC Error: Poly spacing to N-diffusion < 2 lambda at (4, -2)")
        errors.append("DRC Error: Metal1 minimum width violation at (0, 0)")
        
    return errors

## 4. The LangGraph Agents
Watch how the layout coder acts as our "proofreading strategy," taking the specific `drc_errors` and using them to rewrite the `current_tcl_script` to fix placement issues.

In [ ]:
llm = ChatOpenAI(model="gpt-4o", temperature=0.1)

def layout_coder(state: LayoutDesignState):
    """Generates Magic TCL commands to draw layout, fixing DRC errors if any exist."""
    target = state["target_component"]
    errors = state.get("drc_errors", [])
    
    system_prompt = f"""
    You are an expert VLSI Layout Engineer using Magic. Your target is to draw: {target}.
    Here are the current DRC errors from the last script run:
    {errors}
    
    Write a TCL script to fix these errors by moving boxes or resizing them. 
    Output ONLY valid TCL code.
    """
    
    response = llm.invoke([SystemMessage(content=system_prompt)])
    script = response.content.strip().replace("```tcl", "").replace("```", "")
    
    return {
        "current_tcl_script": script,
        "iterations": state.get("iterations", 0) + 1,
        "status": "TCL Script Generated."
    }

def drc_checker(state: LayoutDesignState):
    """Runs the layout script through Magic (Mocked here)."""
    script = state["current_tcl_script"]
    print(f"\n--- Iteration {state['iterations']} ---")
    print("Running DRC Check in Magic VLSI...")
    
    errors = mock_magic_drc(script)
    if len(errors) == 0:
        print("✅ DRC Clean!")
    else:
        print(f"❌ Found {len(errors)} DRC Errors:")
        for e in errors: print(f"  - {e}")
        
    return {
        "drc_errors": errors,
        "status": "DRC Check Complete."
    }

def check_drc_clean(state: LayoutDesignState):
    """Conditional Edge: Loop back if there are errors."""
    if len(state["drc_errors"]) == 0:
        print("\nLayout is DRC Clean. Graph Finishing.")
        return "end"
    elif state["iterations"] >= 3:
        print("\nMax iterations reached. Could not fix DRC.")
        return "end"
    else:
        print("\nLooping back to Coder to fix errors.")
        return "continue"

## 5. Compile the Workflow

In [ ]:
workflow = StateGraph(LayoutDesignState)

workflow.add_node("coder", layout_coder)
workflow.add_node("drc_tool", drc_checker)

workflow.set_entry_point("coder")
workflow.add_edge("coder", "drc_tool")

workflow.add_conditional_edges(
    "drc_tool",
    check_drc_clean,
    {
        "continue": "coder",
        "end": END
    }
)

app = workflow.compile()

# -- Uncomment below to run if you have an API key --
# initial_state = {
#     "target_component": "A basic NMOS transistor layout in 130nm process",
#     "iterations": 0
# }
# final_state = app.invoke(initial_state)
# print("\nFINAL TCL SCRIPT:\n")
# print(final_state["current_tcl_script"])